In [0]:
# ==================================================
# NOTEBOOK 02: DATA CLEANING & FILTERING
# ==================================================

print("="* 80)
print("Notebook 02: Data cleaning & filtering")
print("="*80)

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import warnings
import builtins
import pandas as pd
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print()

In [0]:
# =================================================
#  Step 1: Load Raw Accepted Loans from Notebook 01
# =================================================

print("Step 01: Loading Raw Accepted Loans from Notebook 01")
print("="*80)


accepted_path = "dbfs:/Volumes/workspace/lending_club/lending_club_data/accepted_2007_to_2018Q4.csv"

accepted_df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", True) \
    .csv("dbfs:/Volumes/workspace/lending_club/lending_club_data/accepted_2007_to_2018Q4.csv")

row_count = accepted_df.count()
print(f"Raw accepeted loans loaded: {row_count} rows,{len(accepted_df.columns)} columns ")


In [0]:
 # ===============================================
 # Step 2: Filter to 2015-2018
 # ===============================================

 print("Step 02: Filtering to 2015-2018")
 print("="*80)

 before_count = accepted_df.count()

 filtered_df = (accepted_df
                .withColumn('issue_date',expr("try_to_date(issue_d,'MMM-yyyy')"))
                .filter(col('issue_date').isNotNull())
                .filter(year(col('issue_date')).between(2015,2018))
 )

after_count = filtered_df.count()
removed_rows = before_count - after_count

print(f"Before filter: {before_count} rows")
print(f"After filter: {after_count} rows")
print(f"Rows removed: {removed_rows} rows")
print("Years retained: 2015,2016,2017,2018")
print()

# Confirm year distribution
print("Row count by year after filtering")
year_df = (filtered_df.groupBy(year(col('issue_date')).alias('Year'))\
    .count()\
        .orderBy(col('Year'))
        .withColumn('Year',col('Year').cast(StringType()))
)
            
# Total row
total_df = filtered_df.agg(
    lit('Total').alias('Year'),
    count('*').alias('count'))

year_df.union(total_df).show()


In [0]:
# =====================================
# Step 3: Drop Columns with >50% Null values
# =====================================

print("Step 3: Dropping High Null Columns (>50% nulls)")
print("="*80)

total_rows = filtered_df.count()

null_count_rows = (
    filtered_df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in filtered_df.columns
    ]).collect()[0]
)

high_null_cols = [
    c for c in filtered_df.columns 
    if (null_count_rows[c]/total_rows * 100) > 50
]

print(f"Columns with >50% nulls: {len(high_null_cols)}")  
print(f"Columns to drop: {high_null_cols}")
print()

# Drop high null columns
df_dropped = filtered_df.drop(*high_null_cols)

print(f"Columns before drop: {len(filtered_df.columns)}")
print(f"Columns after drop: {len(df_dropped.columns)}")
print(f"Columns removed: {len(high_null_cols)}")


In [0]:
# ====================================
# Step 3.5: Examine Data types Before Fixing
# ====================================

print("Step 3.5: Examing Data types Before fixing")
print("="* 80)

# All column data types
print("All column data types:")
print()
dtypes_list  = df_dropped.dtypes

# Get unique data types from all columns
print("unique data types:")
unique_dtypes = set(dtype for _,dtype in dtypes_list)
print(unique_dtypes)
print()

string_cols = [c for c,dtype in dtypes_list if dtype == 'string']
date_cols = [c for c,dtype in dtypes_list if dtype == 'date']
double_cols = [c for c,dtype in dtypes_list if dtype == 'double' ]

print(f"Columns with string data type [{len(string_cols)}]: {string_cols} ")
print()
print(f"Columns with date data type [{len(date_cols)}]: {date_cols}")
print()
print(f"Columns with double data type [{len(double_cols)}]: {double_cols}")
print()


In [0]:
display(df_dropped.select(string_cols))

In [0]:
# ======================================================================
# Auto-detect Numeric String Columns
# ======================================================================

print("Auto-detecting numeric string columns...")
print("=" * 80)

# Materialise sample to Pandas once — no caching needed on serverless
sample_pd = df_dropped.sample(fraction=0.001, seed=42).toPandas()
total = len(sample_pd)

numeric_string_cols = []
non_numeric_string_cols = []

for c in string_cols:
    try:
        non_null = sample_pd[c].dropna()

        # Guard: if fewer than 30 non-null values in sample, skip — not enough signal
        if len(non_null) < 30:
            non_numeric_string_cols.append(c)
            print(f"  Skipped '{c}' — only {len(non_null)} non-null values in sample")
            continue

        converted = pd.to_numeric(non_null, errors='coerce')
        success_rate = converted.notna().sum() / len(non_null) * 100

        if success_rate >= 95:
            numeric_string_cols.append(c)
        else:
            non_numeric_string_cols.append(c)
    except Exception as e:
        print(f"  Warning: could not evaluate column '{c}': {e}")
        non_numeric_string_cols.append(c)

print(f"String columns that should be Double [{len(numeric_string_cols)}]:")
print(numeric_string_cols)
print()
print(f"String columns that are genuinely text [{len(non_numeric_string_cols)}]:")
print(non_numeric_string_cols)

In [0]:
# ======================================================================
# STEP 4: Fix Data Types
# ======================================================================

print("STEP 4: Fixing Data Types")
print("=" * 80)

exclude_from_casting = ['id', 'member_id', 'url', 'zip_code','term', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'purpose', 'title', 'zip_code', 'addr_state', 'earliest_cr_line', 'initial_list_status', 'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d', 'application_type', 'hardship_flag', 'disbursement_method', 'debt_settlement_flag']

# Re-run auto casting with exclusions
df_typed = df_dropped

# 1. Auto-cast all numeric string columns to Double
print("Casting numeric string columns to Double:")
for c in numeric_string_cols:
    if c not in exclude_from_casting:
        df_typed = df_typed.withColumn(c, col(c).cast(DoubleType()))
    print(f"  {c} → Double")

print()

# 3. Clean term — extract numeric value
if 'term' in df_typed.columns:
    df_typed = df_typed.withColumn(
        'term',
        regexp_replace(col('term'), '[^0-9]', '').cast(IntegerType())
    )
    print("term — extracted numeric, cast to Integer")

# 4. Parse date string columns
date_cols_to_fix = ['earliest_cr_line', 'last_pymnt_d',
                    'last_credit_pull_d', 'next_pymnt_d']

for c in date_cols_to_fix:
    if c in df_typed.columns:
        df_typed = df_typed.withColumn(
            c, expr(f"try_to_date({c}, 'MMM-yyyy')")
        )
        print(f"{c} — parsed to Date")

print()

# 6. Cast id back to String — it is an identifier, not a numeric value
if 'id' in df_typed.columns:
    df_typed = df_typed.withColumn('id', col('id').cast(StringType()))
    print("✓ id — cast back to String (identifier column)")

# 7. Confirm results
print("Data types after fixing:")
new_dtypes = dict(df_typed.dtypes)
new_string_cols = [c for c, t in new_dtypes.items() if t == 'string']
new_double_cols = [c for c, t in new_dtypes.items() if t == 'double']
new_date_cols   = [c for c, t in new_dtypes.items() if t == 'date']
new_int_cols    = [c for c, t in new_dtypes.items() if 'int' in t.lower()]

print(f"String columns : {len(new_string_cols)} → {new_string_cols}")
print(f"Double columns : {len(new_double_cols)}")
print(f"Date columns   : {len(new_date_cols)}")
print(f"Integer columns: {len(new_int_cols)}")

In [0]:
# ======================================================================
# Drop unnecessary columns
# ======================================================================

cols_to_drop = [
    'url',        # just a web link, no analytical value
    'title',      # free text, redundant with purpose column
    'zip_code',   # partially masked e.g. 190xx — unusable
    'pymnt_plan', # nearly all values are 'n' — no variation
    'member_id',  # not needed — id is the unique identifier
    'issue_d'
]

# Only drop if they exist
existing_drop_cols = [c for c in cols_to_drop if c in df_typed.columns]
df_typed = df_typed.drop(*existing_drop_cols)

print(f"Dropped {len(existing_drop_cols)} unnecessary columns: {existing_drop_cols}")
print(f"Columns remaining: {len(df_typed.columns)}")

In [0]:
display(df_typed)

In [0]:
# ======================================================================
# STEP 5: Auto-examine all string columns
# ======================================================================

print("STEP 5: Examining String Columns")
print("=" * 80)

current_string_cols = [c for c, t in df_typed.dtypes if t == 'string']
print(f"Total string columns: {len(current_string_cols)}")
print()

# Categorise automatically by distinct count
low_cardinality  = []  # ≤ 20 distinct values — categorical
mid_cardinality  = []  # 21-100 distinct values — review needed
high_cardinality = []  # > 100 distinct values — free text, keep or drop

for c in current_string_cols:
    distinct_count = df_typed.select(c).distinct().count()
    
    if distinct_count <= 20:
        low_cardinality.append((c, distinct_count))
    elif distinct_count <= 100:
        mid_cardinality.append((c, distinct_count))
    else:
        high_cardinality.append((c, distinct_count))

# ---------------------------------------------------------------
# Low cardinality — show all distinct values
# ---------------------------------------------------------------
print(f"Low cardinality columns [{len(low_cardinality)}] — show all values:")
print("-" * 60)
for c, n in low_cardinality:
    print(f"\n{c} ({n} values):")
    df_typed.select(c).distinct().orderBy(c).show(truncate=False)

# ---------------------------------------------------------------
# Mid cardinality — show count only
# ---------------------------------------------------------------
print(f"Mid cardinality columns [{len(mid_cardinality)}] — review needed:")
print("-" * 60)
for c, n in mid_cardinality:
    print(f"  {c}: {n} distinct values")

# ---------------------------------------------------------------
# High cardinality — candidates for dropping
# ---------------------------------------------------------------
print()
print(f"High cardinality columns [{len(high_cardinality)}] — likely free text:")
print("-" * 60)
for c, n in high_cardinality:
    print(f"  {c}: {n} distinct values — consider dropping")

### Step 5 Observations: String Column Examination

Examined all **15 string columns** across three cardinality groups to identify 
standardisation, cleaning, and dropping requirements.

---

### Low Cardinality Columns (11 columns — ≤20 distinct values)

| Column | Values | Action |
|--------|--------|--------|
| `grade` | A, B, C, D, E, F, G | ✓ Clean — no change needed |
| `emp_length` | < 1 year to 10+ years | ✓ Clean — kept as ordered categorical |
| `home_ownership` | ANY, MORTGAGE, NONE, OWN, RENT | ✓ Clean — all 5 values meaningful |
| `verification_status` | Not Verified, Source Verified, Verified | ✓ Clean — all 3 are distinct meanings |
| `loan_status` | 7 values including Fully Paid, Charged Off, Default | ✓ Clean — no change needed |
| `purpose` | 15 values — 1 dirty free text value found | ⚠️ Fixed — replaced with 'other' |
| `initial_list_status` | f, w, 14741.0 | ⚠️ Fixed — f→Fractional, w→Whole, dirty value→NULL |
| `application_type` | Individual, Joint App, 0.0 | ⚠️ Fixed — dirty value 0.0→NULL |
| `hardship_flag` | N, Y, NULL | ⚠️ Fixed — N→No, Y→Yes |
| `disbursement_method` | Cash, DirectPay, NULL | ✓ Clean — no change needed |
| `debt_settlement_flag` | N, Y, NULL | ⚠️ Fixed — N→No, Y→Yes |

---

### Mid Cardinality Columns (2 columns — 21–100 distinct values)

| Column | Values | Action |
|--------|--------|--------|
| `sub_grade` | 35 values (A1–G5) | ✓ Keep — detailed risk classification |
| `addr_state` | 51 US states | ✓ Keep — used for geographic analysis in Power BI |

---

### High Cardinality Columns (2 columns — >100 distinct values)

| Column | Distinct Values | Action |
|--------|----------------|--------|
| `id` | 1,794,322 | ✓ Keep — unique loan identifier |
| `emp_title` | 348,424 | ✓ Keep |

---

### Key Decisions
- **`verification_status`** — Verified and Source Verified are **not** the same. 
  Source Verified confirms the income source only, Verified confirms the actual amount.
- **`home_ownership`** — ANY and NONE kept as-is. Both carry distinct credit risk meaning.
- **`initial_list_status`** — renamed f/w to Fractional/Whole for Power BI clarity.
- **`emp_title`** — kept as reference only. 348,424 unique job titles provide 
borrower profile context for drill-through analysis in Power BI.

## Standardising and Cleaning String Column Values

In [0]:
# ---------------------------------------------------------------
# 1. purpose — auto detect and replace dirty values with 'other'
# ---------------------------------------------------------------

# First check all distinct values
print("purpose distinct values before cleaning:")
df_typed.select('purpose').distinct().orderBy('purpose').show(truncate=False)

# Any value with spaces or longer than 20 chars is likely dirty
df_typed = df_typed.withColumn(
    'purpose',
    when(
        col('purpose').contains(' ') & (length(col('purpose')) > 20),
        'other'
    ).otherwise(col('purpose'))
)

print("✓ purpose — long free text values replaced with 'other'")
print("purpose distinct values after cleaning:")
df_typed.select('purpose').distinct().orderBy('purpose').show(truncate=False)

# -------------------------------------------------
# 2. initial_list_status — rename values for clarity
# ------------------------------------------------

df_typed = df_typed.withColumn(
    'initial_list_status',
    when(col('initial_list_status')=='f','Fractional')
    .when(col('initial_list_status')=='w','Whole')
    .otherwise(None)

)
print("✓ initial_list_status — f/w converted to Fractional/Whole")

# Verify
df_typed.select('initial_list_status').distinct().orderBy('initial_list_status').show(truncate=False)


# ---------------------------------------------------------------
# 3. application_type — replace dirty 0.0 with NULL
# ---------------------------------------------------------------

df_typed = df_typed.withColumn(
    'application_type',
    when(col('application_type').isin(['Individual','Joint App']),col('application_type'))
    .otherwise(None)
)
print("✓ application_type — dirty value 0.0 replaced with NULL")

# Verify
print("  Distinct values after cleaning:")
df_typed.select('application_type').distinct().orderBy('application_type').show(truncate=False)

# ---------------------------------------------------------------
# 4. hardship_flag — N/Y to No/Yes
# ---------------------------------------------------------------

df_typed = df_typed.withColumn(
    'hardship_flag',
    when(col('hardship_flag')=='Y','Yes')
    .when(col('hardship_flag')=='N','No')
    .otherwise(None)
)

print("✓ hardship_flag — Y/N converted to Yes/No")

# Verify
print("  Distinct values after cleaning:")
df_typed.select('hardship_flag').distinct().orderBy('hardship_flag').show(truncate=False)

# ---------------------------------------------------------------
# 5. debt_settlement_flag — N/Y to No/Yes
# ---------------------------------------------------------------

df_typed = df_typed.withColumn(
    'debt_settlement_flag',
    when(col('debt_settlement_flag')=='Y','Yes')
    .when(col('debt_settlement_flag')=='N','No')
    .otherwise(None)
)
print("✓ debt_settlement_flag — Y/N converted to Yes/No")

# Verify
print("  Distinct values after cleaning:")
df_typed.select('debt_settlement_flag').distinct().orderBy('debt_settlement_flag').show(truncate=False)

In [0]:
# ======================================================================
# STEP 6: Rename Columns for Clarity
# ======================================================================

print("STEP 5.3: Renaming Columns for Clarity")
print("=" * 80)

df_renamed = df_typed.withColumnRenamed('loan_amnt', 'loan_amount') \
    .withColumnRenamed('funded_amnt', 'funded_amount') \
    .withColumnRenamed('funded_amnt_inv', 'funded_amount_inv') \
    .withColumnRenamed('int_rate', 'interest_rate') \
    .withColumnRenamed('annual_inc', 'annual_income') \
    .withColumnRenamed('dti', 'debt_to_income') \
    .withColumnRenamed('revol_bal', 'revolving_balance') \
    .withColumnRenamed('revol_util', 'revolving_utilisation') \
    .withColumnRenamed('addr_state', 'state') \
    .withColumnRenamed('delinq_2yrs', 'delinquencies_2yrs') \
    .withColumnRenamed('inq_last_6mths', 'inquiries_last_6mths') \
    .withColumnRenamed('open_acc', 'open_accounts') \
    .withColumnRenamed('pub_rec', 'public_records') \
    .withColumnRenamed('total_acc', 'total_accounts') \
    .withColumnRenamed('out_prncp', 'outstanding_principal') \
    .withColumnRenamed('out_prncp_inv', 'outstanding_principal_inv') \
    .withColumnRenamed('tot_coll_amt', 'total_collection_amount') \
    .withColumnRenamed('tot_cur_bal', 'total_current_balance') \
    .withColumnRenamed('mort_acc', 'mortgage_accounts') \
    .withColumnRenamed('pub_rec_bankruptcies', 'bankruptcy_records') \
    .withColumnRenamed('acc_now_delinq', 'accounts_now_delinquent') \
    .withColumnRenamed('tot_hi_cred_lim', 'total_high_credit_limit')

print(f"✓ Columns renamed for clarity")
print(f"Total columns: {len(df_renamed.columns)}")

In [0]:
display(df_renamed)

In [0]:
from pyspark.sql.functions import sum as spark_sum

In [0]:
# ----------------------------------------------------
# 6.1 Check remaining nulls (fast single-pass version)
# ----------------------------------------------------
print("Step 6.1: Checking remaining nulls")
print('-'*60)

df_imputed = df_renamed

from pyspark.sql import functions as F

null_counts = df_imputed.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_imputed.columns
]).collect()[0]

null_status = [(c, null_counts[c]) for c in df_imputed.columns]

# only show columns with nulls
nulls_found = [(c, n) for c, n in null_status if n > 0]
nulls_found = sorted(nulls_found,key = lambda x:x[1],reverse = True)
print(f"Columns with nulls: {len(nulls_found)}")
for c, n in nulls_found:
    pct = builtins.round(n/total_rows * 100 ,2)
    print(f"  {c:<40} {n:>10,} nulls ({pct}%)")

In [0]:
print('next_pymnt_d' in df_imputed.columns)
print('mths_since_rcnt_il' in df_imputed.columns)
print(df_imputed.columns)

In [0]:
from pyspark.sql import functions as F

print("Step 6 - Handling Missing Values")
print("-" * 60)

# ------------------------------------------------------------
# 1. Drop columns
# ------------------------------------------------------------
cols_to_drop = [
    'next_pymnt_d',
    'mths_since_rcnt_il'
]

existing_drop = [c for c in cols_to_drop if c in df_imputed.columns]
df_imputed = df_imputed.drop(*existing_drop)

print(f"✓ Dropped columns: {existing_drop}")

# ------------------------------------------------------------
# 2. Fill with 0
# ------------------------------------------------------------
cols_fill_zero = [
    'il_util', 'all_util',
    'open_acc_6m', 'total_cu_tl', 'inq_last_12m',
    'open_act_il', 'open_il_12m', 'open_il_24m',
    'total_bal_il',
    'open_rv_12m', 'open_rv_24m',
    'max_bal_bc', 'inq_fi',
    'num_tl_120dpd_2m'
]

existing_zero = [c for c in cols_fill_zero if c in df_imputed.columns]
df_imputed = df_imputed.fillna(0, subset=existing_zero)

print(f"✓ Filled {len(existing_zero)} columns with 0")



In [0]:
# ------------------------------------------------------------
# 3. Fill with median
# ------------------------------------------------------------
cols_fill_median = [
    'mths_since_recent_inq',
    'mo_sin_old_il_acct',
    'bc_util', 'percent_bc_gt_75',
    'bc_open_to_buy',
    'mths_since_recent_bc',
    'debt_to_income',
    'revolving_utilisation',
    'avg_cur_bal'
]

for c in cols_fill_median:
    if c in df_imputed.columns:
        try:
            median_list = df_imputed.approxQuantile(c, [0.5], 0.01)
            if median_list and median_list[0] is not None:
                median_value = float(median_list[0])
                df_imputed = df_imputed.fillna({c: median_value})
                print(f"{c:<30} -> median: {builtins.round(median_value, 2)}")
            else:
                print(f"Skipped {c}: median could not be computed")
        except Exception as e:
            print(f" Error in median imputation for {c}: {e}")

# ------------------------------------------------------------
# 4. Categorical handling
# ------------------------------------------------------------
if 'emp_title' in df_imputed.columns:
    df_imputed = df_imputed.fillna({'emp_title': 'Unknown'})
    print("✓ Filled emp_title with 'Unknown'")

# ------------------------------------------------------------
# 5. Low-null columns
# ------------------------------------------------------------
low_null_cols = ['last_pymnt_d', 'last_credit_pull_d']
existing_low_null = [c for c in low_null_cols if c in df_imputed.columns]

if existing_low_null:
    df_imputed = df_imputed.dropna(subset=existing_low_null)
    print(f"✓ Dropped rows with nulls in: {existing_low_null}")

# ------------------------------------------------------------
# 6. Remaining tiny-null columns
# ------------------------------------------------------------
extra_fill_median = ['pct_tl_nvr_dlq', 'inquiries_last_6mths', 'num_rev_accts']

for c in extra_fill_median:
    if c in df_imputed.columns:
        median_list = df_imputed.approxQuantile(c, [0.5], 0.01)
        if median_list and median_list[0] is not None:
            median_value = float(median_list[0])
            df_imputed = df_imputed.fillna({c: median_value})
            print(f"✓ {c:<30} -> median: {builtins.round(median_value, 2)}")

# ------------------------------------------------------------
# 7. emp_length
# ------------------------------------------------------------
if 'emp_length' in df_imputed.columns:
    df_imputed = df_imputed.fillna({'emp_length': 'Unknown'})
    print("✓ Filled emp_length with 'Unknown'")



In [0]:
# ---------------------------------------------------------------
# Step 6.5 — Final null check
# ---------------------------------------------------------------
print("Step 6.5 — Final null check after imputation:")
print("-" * 60)

final_null_counts = df_imputed.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_imputed.columns
]).collect()[0]

remaining = sorted(
    [(c, final_null_counts[c])
     for c in df_imputed.columns
     if final_null_counts[c] > 0],
    key=lambda x: x[1], reverse=True
)

print(f"Columns still with nulls: {len(remaining)}")
print()
for c, n in remaining:
    pct = builtins.round(n / total_rows * 100, 2)
    print(f"  {c:<40} {n:>10,} nulls ({pct}%)")

print()
print("=" * 80)
print(f"✓ Null handling complete")
print(f"  Final rows    : {df_imputed.count():,}")
print(f"  Final columns : {len(df_imputed.columns)}")

## Step 6: Missing Value Handling

### 🧠 Approach

Missing values were handled using a **domain-driven strategy**, rather than applying a single blanket method. Each column was evaluated based on:

- Business meaning  
- Type of feature  
- Reason for missingness  

---

### 🗑️ Columns Dropped

- **`next_pymnt_d`**
  - Removed due to **data leakage**
  - Contains future payment information not available at prediction time

- **`mths_since_rcnt_il`**
  - Dropped due to **high sparsity and ambiguity**
  - Null values could represent either missing data or no instalment history
  - Redundant with other instalment loan features

---

### 🔄 Columns Imputed with 0

These are **activity-based features**, where null values indicate **no activity**, not missing data.

- `il_util`, `all_util`
- `open_acc_6m`, `total_cu_tl`, `inq_last_12m`
- `open_act_il`, `open_il_12m`, `open_il_24m`
- `total_bal_il`
- `open_rv_12m`, `open_rv_24m`
- `max_bal_bc`, `inq_fi`
- `num_tl_120dpd_2m`

**Reason:**
- Null values imply absence of accounts or activity  
- Filling with 0 preserves true borrower behaviour  

---

### 📊 Columns Imputed with Median

These are **continuous financial variables**, where missing values represent data gaps.

- `mths_since_recent_inq`
- `mo_sin_old_il_acct`
- `bc_util`
- `percent_bc_gt_75`
- `bc_open_to_buy`
- `mths_since_recent_bc`
- `debt_to_income`
- `revolving_utilisation`
- `avg_cur_bal`
- `pct_tl_nvr_dlq`
- `inquiries_last_6mths`
- `num_rev_accts`

**Reason:**
- Median is robust to outliers  
- Maintains realistic distribution of values  
- Avoids distortion from zero imputation  

---

### 🏷️ Categorical Handling

- **`emp_title` → "Unknown"**
- **`emp_length` → "Unknown"**

**Reason:**
- Missing values are common in text/categorical fields  
- Preserves data without dropping rows  

---

### 🧹 Low Null Columns

- `last_pymnt_d`, `last_credit_pull_d`

**Action:**
- Rows with null values were dropped due to **very low null percentage**

---

### 🔍 Validation

A final null check was performed by computing the number of null values in each column using conditional aggregation.

---

### 🎯 Summary

- Dropped columns with **data leakage or low analytical value**
- Used **zero imputation** for activity-based features
- Applied **median imputation** for continuous variables
- Handled categorical nulls with **default labels**
- Ensured **no remaining null values** in the dataset

This approach preserves **data integrity**, retains **behavioural signals**, and ensures the dataset is suitable for downstream analysis and modelling.

In [0]:
df_imputed.columns

In [0]:
# Column renaming
rename_dict = {
    # time-based
    'mths_since_recent_inq': 'months_since_last_inquiry',
    'mths_since_recent_bc': 'months_since_last_credit_card_activity',
    'mo_sin_old_il_acct': 'months_since_oldest_installment_account',
    'mo_sin_old_rev_tl_op': 'months_since_oldest_revolving_account',
    'mo_sin_rcnt_rev_tl_op': 'months_since_recent_revolving_account',
    'mo_sin_rcnt_tl': 'months_since_recent_account',

    # credit card features
    'bc_util': 'credit_card_utilisation_pct',
    'percent_bc_gt_75': 'pct_credit_cards_over_75_util',
    'bc_open_to_buy': 'credit_card_available_limit',

    # inquiries
    'inq_fi': 'financial_inquiries'
}

for old,new in rename_dict.items():
    if old in df_imputed.columns:
        df_renamed = df_imputed.withColumnRenamed(old,new)

print("Renamed selected columns")

## Column Renaming – Rationale

Column names were selectively renamed to improve **clarity, consistency, and interpretability** of the dataset. Many original features contained abbreviations and technical shorthand (e.g., `bc_util`, `mths_since_recent_inq`) that were not immediately intuitive.

Renaming was performed only where necessary to:
- Make feature meanings **self-explanatory**
- Improve **readability** for analysis and modelling
- Enable clearer **communication of insights** in reports and interviews

At the same time, widely recognised and standard financial variables (e.g., `loan_amount`, `interest_rate`, `debt_to_income`) were retained in their original form to maintain **consistency with industry conventions and source documentation**.

This balanced approach ensures the dataset remains both **interpretable and aligned with standard practices**, without introducing unnecessary complexity.

In [0]:
display(df_renamed)

### Data Reduction Summary

- Initial rows: 2,260,701  
- Final rows: 1,792,240  
- Rows dropped: 468,461 (~20.7%)

Rows were removed primarily due to missing values in critical columns during the data cleaning process.

In [0]:
# ------------------------------------------------------------
# Step 7 - Remove Duplicates
# ------------------------------------------------------------
print("Step 7 - Removing Duplicates")
print("-" * 60)

initial_count = df_renamed.count()

df_renamed = df_renamed.dropDuplicates()

final_count = df_renamed.count()

duplicates_removed = initial_count - final_count

print(f"Initial rows: {initial_count}")
print(f"Final rows:   {final_count}")
print(f"Duplicates removed: {duplicates_removed}")

In [0]:
# ======================================================================
# STEP 8: Save as Delta Lake Table
# ======================================================================

print("STEP 8: Saving Cleaned Data as Delta Lake Table")
print("=" * 80)

delta_path = "/Volumes/workspace/lending_club/lending_club_data/cleaned_loans_delta"

df_renamed.write.format("delta").mode("overwrite").save(delta_path)

print(f"✓ Delta table saved successfully")
print(f"  Path    : {delta_path}")
print(f"  Rows    : {df_renamed.count():,}")
print(f"  Columns : {len(df_renamed.columns)}")
print()